In [4]:
import pandas as pd
import numpy as np
import geopandas as gpd
import xarray as xr
import pystac_client
import planetary_computer
import odc.stac
from shapely.geometry import Point


In [5]:
hh = pd.read_csv("../data/target_data_pipeline.csv")

hh_unique = hh[['hhid','gps_longitude','gps_latitude']].drop_duplicates()

hh_gdf = gpd.GeoDataFrame(
    hh_unique,
    geometry=gpd.points_from_xy(
        hh_unique.gps_longitude,
        hh_unique.gps_latitude
    ),
    crs="EPSG:4326"
)

lon_vals = hh_gdf.geometry.x.values
lat_vals = hh_gdf.geometry.y.values
hh_ids = hh_gdf['hhid'].values

MY_BBOX = [
    lon_vals.min(),
    lat_vals.min(),
    lon_vals.max(),
    lat_vals.max(),
]


In [6]:
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

collection = catalog.get_collection("terraclimate")
asset = collection.assets["zarr-abfs"]
open_kwargs = asset.extra_fields["xarray:open_kwargs"]

ds_tc = xr.open_dataset(asset.href, **open_kwargs)

ds_tc = ds_tc.sel(
    lon=slice(MY_BBOX[0], MY_BBOX[2]),
    lat=slice(MY_BBOX[3], MY_BBOX[1]),
    time=slice("2008-01-01", "2022-12-31")
)

ds_tc = ds_tc[["soil", "ppt", "pdsi", "vpd"]]
ds_tc = ds_tc.load()





In [7]:
tc_dfs = []

for i, (idx, row) in enumerate(hh_gdf.iterrows()):

    if i % 100 == 0:
        print(f"Processed {i} households")

    hhid = row['hhid']
    lon = row.geometry.x
    lat = row.geometry.y

    # Extract nearest grid cell indices first
    lon_idx = np.abs(ds_tc.lon.values - lon).argmin()
    lat_idx = np.abs(ds_tc.lat.values - lat).argmin()

    # Extract time series as numpy arrays
    soil_vals = ds_tc.soil[:, lat_idx, lon_idx].values
    ppt_vals = ds_tc.ppt[:, lat_idx, lon_idx].values
    pdsi_vals = ds_tc.pdsi[:, lat_idx, lon_idx].values
    vpd_vals = ds_tc.vpd[:, lat_idx, lon_idx].values
    time_vals = ds_tc.time.values

    df = pd.DataFrame({
        "hhid": hhid,
        "month": pd.to_datetime(time_vals),
        "soil": soil_vals,
        "ppt": ppt_vals,
        "pdsi": pdsi_vals,
        "vpd": vpd_vals
    })

    tc_dfs.append(df)

df_tc = pd.concat(tc_dfs, ignore_index=True)


Processed 0 households
Processed 100 households
Processed 200 households
Processed 300 households
Processed 400 households
Processed 500 households
Processed 600 households
Processed 700 households
Processed 800 households
Processed 900 households
Processed 1000 households


In [8]:
modis = pd.read_csv("../data/modis_household_monthly.csv")
modis["month"] = pd.to_datetime(modis["month"])

monthly_pred = df_tc.merge(
    modis,
    on=["hhid","month"],
    how="left"
)


In [9]:
monthly_pred = monthly_pred.sort_values(
    ["hhid", "month"]
).reset_index(drop=True)


In [11]:
predictor_cols = [
    "soil", "ppt", "pdsi", "vpd",
    "ndvi", "lai", "lst"
]


In [12]:
monthly_pred[predictor_cols] = (
    monthly_pred
    .groupby("hhid")[predictor_cols]
    .transform(lambda x: x.interpolate())
)

monthly_pred[predictor_cols] = (
    monthly_pred
    .groupby("hhid")[predictor_cols]
    .transform(lambda x: x.ffill().bfill())
)


In [15]:
lags = [1, 2, 3]

for col in predictor_cols:
    for lag in lags:
        monthly_pred[f"{col}_lag{lag}"] = (
            monthly_pred
            .groupby("hhid")[col]
            .shift(lag)
        )


In [21]:
monthly_pred

,hhid,month,soil,ppt,pdsi,vpd,ndvi,soil_lag1,soil_lag2,soil_lag3,...,vpd_lag2,vpd_lag3,ndvi_lag1,ndvi_lag2,ndvi_lag3,soil_roll3,ppt_roll3,pdsi_roll3,vpd_roll3,ndvi_roll3
3,1002,2008-04-01,2.9,74.5,-2.24,1.36,0.62050,3.1,3.3,3.5,...,2.48,2.55,0.40515,0.44710,0.49610,3.100000,47.966667,-2.183333,1.903333,0.490917
4,1002,2008-05-01,2.8,21.5,-2.78,1.46,0.67320,2.9,3.1,3.3,...,1.87,2.48,0.62050,0.40515,0.44710,2.933333,55.066667,-2.190000,1.563333,0.566283
5,1002,2008-06-01,2.6,0.0,-2.61,1.56,0.62210,2.8,2.9,3.1,...,1.36,1.87,0.67320,0.62050,0.40515,2.766667,32.000000,-2.543333,1.460000,0.638600
6,1002,2008-07-01,2.5,0.2,-2.44,1.52,0.62190,2.6,2.8,2.9,...,1.46,1.36,0.62210,0.67320,0.62050,2.633333,7.233333,-2.610000,1.513333,0.639067
7,1002,2008-08-01,2.4,0.0,-2.25,1.68,0.65850,2.5,2.6,2.8,...,1.56,1.46,0.62190,0.62210,0.67320,2.500000,0.066667,-2.433333,1.586667,0.634167
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172195,16066,2021-08-01,2.0,2.6,-2.98,1.35,0.73795,2.1,2.3,2.4,...,1.10,0.83,0.66520,0.66985,0.64320,2.133333,5.366667,-2.750000,1.143333,0.691000
172196,16066,2021-09-01,1.9,30.5,-1.85,1.21,0.72715,2.0,2.1,2.3,...,0.98,1.10,0.73795,0.66520,0.66985,2.000000,15.300000,-2.493333,1.180000,0.710100
172197,16066,2021-10-01,1.8,7.8,-2.38,1.41,0.73900,1.9,2.0,2.1,...,1.35,0.98,0.72715,0.73795,0.66520,1.900000,13.633333,-2.403333,1.323333,0.734700
172198,16066,2021-11-01,1.7,21.8,-3.27,1.36,0.69520,1.8,1.9,2.0,...,1.21,1.35,0.73900,0.72715,0.73795,1.800000,20.033333,-2.500000,1.326667,0.720450


In [23]:
import os
import pandas as pd

os.makedirs('predictions', exist_ok=True)
monthly_pred.to_csv('monthly_predictions.csv', index=False)
